In [1]:
!wget https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip
!unzip -q maestro-v3.0.0-midi.zip

--2026-10-14 16:16:29--  https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.132.207, 74.125.69.207, 173.194.206.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.132.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 58416533 (56M) [application/octet-stream]
Saving to: ‘maestro-v3.0.0-midi.zip.1’

maestro-v3.0.0-midi 100%[===================>]  55.71M  69.8MB/s    in 0.8s    

2026-10-14 16:16:30 (69.8 MB/s) - ‘maestro-v3.0.0-midi.zip.1’ saved [58416533/58416533]

replace maestro-v3.0.0/2004/MIDI-Unprocessed_XP_08_R1_2004_01-02_ORIG_MID--AUDIO_08_R1_2004_01_Track01_wav.midi? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [2]:
!apt-get update -qq
!apt-get install -y -qq fluidsynth fluid-soundfont-gm
!pip install -q pyfluidsynth pretty_midi

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [3]:
import fluidsynth


### READING FILES

In [4]:
import pretty_midi
from pathlib import Path
from tqdm import tqdm

files = sorted(Path("maestro-v3.0.0").rglob("*.midi"))
print(len(files))   # should be around 1,276
print(files[0])

1276
maestro-v3.0.0/2004/MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_05_Track05_wav.midi


In [5]:
"""
CALCULATE MIN AND MAX IF NECESSARY
MAX = 0
MIN = float('inf')

for file in tqdm(files):
    pm = pretty_midi.PrettyMIDI(str(file))
    piano = pm.instruments[0]
    for note in piano.notes:
        if MAX < note.pitch: MAX = note.pitch
        if MIN > note.pitch: MIN = note.pitch"""

"\nCALCULATE MIN AND MAX IF NECESSARY\nMAX = 0\nMIN = float('inf')\n\nfor file in tqdm(files):\n    pm = pretty_midi.PrettyMIDI(str(file))\n    piano = pm.instruments[0]\n    for note in piano.notes:\n        if MAX < note.pitch: MAX = note.pitch\n        if MIN > note.pitch: MIN = note.pitch"

In [6]:
MAX = 108
MIN = 21

In [7]:
import numpy as np
import torch

SCALE = 0.001
MAXTIME = 100
pieces = []

class Note():
    def __init__(self, pitch, time, timing):
        self.pitch = pitch
        self.time = int(round(time / SCALE))
        self.timing = timing

def make_tokens(file):
    j = k = 0
    notes = []
    pm = pretty_midi.PrettyMIDI(str(file))
    piano = next(inst for inst in pm.instruments if not inst.is_drum)
    for note in piano.notes:
            note_1 = Note(note.pitch, note.start, 'START')
            note_2 = Note(note.pitch, note.end, 'END')
            notes.append(note_1)
            notes.append(note_2)
    sorted_notes = sorted(notes, key=lambda x: x.time)

    final_notes = []
    for i in range(len(sorted_notes) - 1):
        final_notes.append((sorted_notes[i].timing, sorted_notes[i].pitch))
        time_step = sorted_notes[i+1].time - sorted_notes[i].time
        if time_step != 0:
            while time_step > MAXTIME:
                final_notes.append(('TIME', MAXTIME))
                time_step -= MAXTIME
            final_notes.append(('TIME', time_step))

    final_notes.append((sorted_notes[-1].timing, sorted_notes[-1].pitch))
    return final_notes

In [8]:
token_sqs = [make_tokens(file) for file in tqdm(files)]

100%|██████████| 1276/1276 [08:41<00:00,  2.45it/s]


In [9]:
token_sqs_lens = [len(token_sq) for token_sq in token_sqs]

MAX_PIECE = max(token_sqs_lens)
MIN_PIECE = min(token_sqs_lens)
MEAN_PIECE = sum(token_sqs_lens)/len(token_sqs_lens)

print(f"MAX: {MAX_PIECE} MIN: {MIN_PIECE} MEAN: {MEAN_PIECE}")

MAX: 109069 MIN: 853 MEAN: 24211.17789968652


In [10]:
def make_vocab():
    vocab = {'PAD': 0, 'BOS': 1, 'EOS': 2}
    for kind in ('START', 'END'):
        for p in range(MIN, MAX+1):
            vocab[(kind, p)] = len(vocab)
    for t in range(1, MAXTIME+1):
        vocab[('TIME', t)] = len(vocab)
    return vocab

def make_ids(tokens, vocab):
    ids = [vocab['BOS']]
    for key, val in tokens:
        ids.append(vocab[(key, val)])
    ids.append(vocab['EOS'])
    return ids

In [11]:
#CREATE VOCAB TOKEN -> IDS
vocab = make_vocab()
vocab_size = len(vocab)

In [12]:
#CREATING THE TENSOR
sq_len = 100

sqs = []
for sq in tqdm(token_sqs):
    sq = torch.tensor(make_ids(sq, vocab), dtype=torch.long)[:sq_len]
    sq = torch.nn.functional.pad(sq, (0, sq_len - len(sq)), value=0)
    sqs.append(sq)

data = torch.stack(sqs)

100%|██████████| 1276/1276 [00:07<00:00, 179.31it/s]


In [13]:
n = 0.9

data_train = data[:int(len(data)*n)]
data_test = data[int(len(data)*n):]

In [14]:
from torch.utils.data import DataLoader
def make_xy(samples):
    samples = torch.stack(samples)
    return samples[..., :-1], samples[..., 1:]

data_train_loader = DataLoader(data_train, batch_size=16, shuffle=True, collate_fn=make_xy)
data_test_loader = DataLoader(data_test, batch_size=16, shuffle=True, collate_fn=make_xy)

### BIGRAM MODEL

In [15]:
import torch.nn as nn
torch.manual_seed(1502)

class BigramModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx):
        logits = self.token_embedding_table(idx)
        B, N, L = logits.shape #Inside each batch, each note is now a list of logits
        logits = logits.reshape(B*N, L)
        return logits

In [26]:
m = BigramModel(vocab_size)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=m.parameters(), lr=1e-3)

for epoch in tqdm(range(100)):

    for xb, yb in data_train_loader:

        B, N = yb.shape
        yb = yb.reshape(B*N)

        y_pred = m(xb)
        loss = loss_fn(y_pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
print(loss.item())

100%|██████████| 100/100 [00:28<00:00,  3.46it/s]

3.953007936477661


### AUDIO GENERATION

In [43]:
from IPython.display import Audio, display

def generate(bigrammodel, max_tokens, idx=100): #idx=1=BOS
    sq = [1, idx]
    for _ in range(max_tokens):
        idx = torch.tensor(idx, dtype=torch.long).unsqueeze(dim=-1).unsqueeze(dim=-1)
        logits = m(idx)
        probs = nn.functional.softmax(logits, dim=-1)
        idx = torch.multinomial(probs, num_samples = 1).item()
        sq.append(idx)
    return sq

def ids_to_tokens(ids, vocab):
    inv = {v: k for k, v in vocab.items()}
    tokens = []
    for i in ids:
        tok = inv[int(i)]
        if tok in ('PAD', 'BOS'):
            continue
        if tok == 'EOS':
            break
        tokens.append(tok)
    return tokens

def tokens_to_midi(tokens, scale, path='out.mid', velocity=80):
    pm = pretty_midi.PrettyMIDI()
    inst = pretty_midi.Instrument(program=0)
    t = 0                                # clock, in quantized ticks
    active = {}                          # pitch -> start tick

    def close(pitch, end_tick):
        start = active.pop(pitch)
        if end_tick > start:             # skip zero-length notes
            inst.notes.append(pretty_midi.Note(
                velocity=velocity, pitch=pitch,
                start=start * scale, end=end_tick * scale))

    for kind, val in tokens:
        if kind == 'TIME':
            t += val
        elif kind == 'START':
            if val in active:            # same pitch already sounding
                close(val, t)
            active[val] = t
        elif kind == 'END':
            if val in active:            # ignore ENDs with no matching START
                close(val, t)

    for pitch in list(active):           # close notes left hanging at the end
        close(pitch, t)

    pm.instruments.append(inst)
    pm.write(path)
    return pm

def make_audio(pm):
    sf2 = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
    audio = pm.fluidsynth(fs=44100, sf2_path=sf2)
    display(Audio(audio, rate=44100))

def ids_to_audio(ids, vocab=vocab, scale=SCALE, path='out.mid', velocity=80):
    tokens = ids_to_tokens(ids, vocab)
    pm = tokens_to_midi(tokens, scale, path, velocity)
    make_audio(pm)

In [44]:
ids = generate(bigrammodel=m, max_tokens=1000)
tokens = ids_to_tokens(ids, vocab)
pm = tokens_to_midi(tokens, scale=SCALE)
make_audio(pm)

/tmp/ipykernel_30270/192781678.py:58: DeprecationWarning: The parameter 'sf2_path' is deprecated, please use 'synthesizer' instead.
  audio = pm.fluidsynth(fs=44100, sf2_path=sf2)


In [36]:
tokens

[]